# Deploying Agents with Gradio

*Notebook 32*

Notebooks are private. Gradio makes your agent public — a real chat interface anyone can use, deployed in minutes.
<br>
<br>

**Topics:**

- What Gradio is and why it fits agent demos

- Wrapping an agent in `gr.ChatInterface`

- Streaming responses — bridging `Runner.run_streamed()` to Gradio

- Tool visibility — showing what the agent is doing in real time

- Handling failure states gracefully

- Deploying to Hugging Face Spaces

- Secret management for deployment

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("..") / ".env")

MODEL = "gpt-5-mini"

print("✅ Ready!")

try:
    import gradio as gr
    print("✅ Gradio ready")
except ImportError:
    print("❌ Gradio not found — run: pip install gradio")

---

## 🎯 The Problem

Your agent works in a notebook — but notebooks aren't for users. Gradio gives you a real chat interface with one function, deployable to Hugging Face Spaces in minutes.

---

## 🖥️ Part 1: What Gradio Is

Gradio is a Python library for building web UIs around machine learning models and AI agents. It's the standard tool for AI demos because:

- A working chat interface takes about 10 lines of code

- `gr.ChatInterface` handles the conversation loop, message history, and UI automatically

- Gradio apps deploy directly to Hugging Face Spaces — free hosting, public URL, no infrastructure

---

## 💬 Part 2: Wrap an Agent in `gr.ChatInterface`

These next four parts build the same app step by step — start with Part 2, then layer on streaming, status updates, and error handling. Part 6's deployment template combines them all.

The minimal Gradio app: define a chat function, pass it to `gr.ChatInterface`, launch. Gradio handles everything else.

In [ ]:
# Minimal Gradio chat app
# -----------------------
MINIMAL_APP = '''
import gradio as gr
from agents import Agent, Runner
from config import MODEL

agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant.",
    model=MODEL
)


async def chat(message: str, history: list) -> str:
    """Handle one turn of the conversation."""

    result = await Runner.run(agent, input=message)

    return result.final_output


demo = gr.ChatInterface(
    fn=chat,
    title="My Agent",
    description="Chat with your AI agent."
)

if __name__ == "__main__":
    demo.launch()
'''

print("Minimal Gradio app:")
print(MINIMAL_APP)

### 💡 Why This Works

`gr.ChatInterface` expects a function that takes `message` and `history` and returns a string. Gradio manages the conversation loop, the message display, and the UI. Your agent code doesn't change at all — you're just changing what calls it.

⚠️ **History vs memory:** `gr.ChatInterface` passes `history` to your chat function, but this is UI state — it shows past messages in the interface. The agent itself is stateless unless you reconstruct history into the input or use sessions (Lesson 19). Use this pattern for lightweight short-term context in a simple chat app; use sessions when you want the SDK to manage conversation state for you.

Gradio passes `history` as a list of `{'role': 'user'|'assistant', 'content': text}` dicts (most recent last). To include recent context, build the input from history before passing it to the agent:

    context = "\n".join([f"User: {h['content']}" if h["role"] == "user" else f"Assistant: {h['content']}" for h in history[-6:]])
    full_input = f"{context}\nUser: {message}" if context else message
    result = await Runner.run(agent, input=full_input)

For persistent memory across sessions, use Lesson 20 instead.

---

## 📡 Part 3: Streaming Responses

The minimal app waits for the full response before displaying anything. Streaming makes the experience feel faster and more responsive — text appears as it's generated.

Gradio supports streaming via Python generators: `yield` partial responses instead of `return`ing the full one. Use `Runner.run_streamed()` and pass each text chunk into Gradio with `yield`.

In [ ]:
# Streaming Gradio app
# --------------------
STREAMING_APP = '''
import gradio as gr
from agents import Agent, Runner
from agents.stream_events import RawResponsesStreamEvent
from openai.types.responses import ResponseTextDeltaEvent
from config import MODEL

agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant.",
    model=MODEL
)


async def chat(message: str, history: list):
    """Stream the agent response token by token."""
    response_text = ""

    result = Runner.run_streamed(agent, input=message)

    async for event in result.stream_events():
        if isinstance(event, RawResponsesStreamEvent):
            data = event.data
            if isinstance(data, ResponseTextDeltaEvent):
                response_text += data.delta
                yield response_text  # Gradio updates the UI on each yield


demo = gr.ChatInterface(
    fn=chat,
    title="My Agent (Streaming)",
    description="Responses appear as they arrive."
)

if __name__ == "__main__":
    demo.launch()
'''

print("Streaming Gradio app:")
print(STREAMING_APP)

### 💡 Why This Works

Gradio treats any generator function as a streaming source — every `yield` updates the chat window. `Runner.run_streamed()` produces text deltas as the model generates them. The bridge is simple: accumulate the text, yield the growing string on each delta. Gradio handles the rest.

---

## 🔧 Part 4: Tool Visibility

When an agent calls a tool, the user sees nothing — the UI just waits. Adding a brief status update makes the experience feel responsive and shows that something is happening. This matters most when tools are slow — without a status update, users often assume the app froze or the request failed.

The pattern: yield a status line before the tool result arrives, then replace it with the final response.

In [ ]:
# Streaming with tool visibility
# --------------------------------
TOOL_VISIBILITY_APP = '''
import gradio as gr
from agents import Agent, Runner, function_tool, ToolCallItem
from agents.stream_events import RawResponsesStreamEvent, RunItemStreamEvent
from openai.types.responses import ResponseTextDeltaEvent
from config import MODEL


@function_tool
def get_weather(location: str) -> str:
    """Get current weather for a location."""
    return f"Weather in {location}: 72°F and sunny"


agent = Agent(
    name="WeatherAssistant",
    instructions="Help users with weather questions.",
    model=MODEL,
    tools=[get_weather]
)


async def chat(message: str, history: list):
    """Stream responses and show tool call status."""
    response_text = ""

    result = Runner.run_streamed(agent, input=message)

    async for event in result.stream_events():
        if isinstance(event, RunItemStreamEvent):
            # Tool call starting — show status to user
            item = event.item
            if isinstance(item, ToolCallItem):
                yield f"🔧 Calling {item.name}..."

        elif isinstance(event, RawResponsesStreamEvent):
            data = event.data
            if isinstance(data, ResponseTextDeltaEvent):
                response_text += data.delta
                yield response_text


demo = gr.ChatInterface(
    fn=chat,
    title="Weather Agent",
    description="Ask about the weather anywhere."
)

if __name__ == "__main__":
    demo.launch()
'''

print("Tool visibility app:")
print(TOOL_VISIBILITY_APP)

### 💡 Why This Works

`RunItemStreamEvent` fires for various agent activities, including when a tool call starts. Yielding a status string at that point surfaces tool progress in the chat window, which is then replaced by the actual response as text deltas arrive.

---

## ⚠️ Part 5: Handling Failure States

Agent calls can fail — tool timeouts, API errors, or model issues. A Gradio app that crashes with a stack trace feels broken — a Gradio app that returns a friendly message feels like a product.

Wrap the agent call in a try/except and yield a clear error message when something goes wrong.

In [ ]:
# Streaming app with failure handling
# -------------------------------------
FAILURE_HANDLING_APP = '''
import gradio as gr
from agents import Agent, Runner
from agents.stream_events import RawResponsesStreamEvent
from openai.types.responses import ResponseTextDeltaEvent
from config import MODEL

agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant.",
    model=MODEL
)


async def chat(message: str, history: list):
    """Stream responses with graceful error handling."""
    try:
        response_text = ""

        result = Runner.run_streamed(agent, input=message)

        async for event in result.stream_events():
            if isinstance(event, RawResponsesStreamEvent):
                data = event.data
                if isinstance(data, ResponseTextDeltaEvent):
                    response_text += data.delta
                    yield response_text

    except Exception as e:
        # Return a friendly message — never expose raw exceptions to users
        yield f"⚠️ Something went wrong. Please try again. (Error: {type(e).__name__})"


demo = gr.ChatInterface(
    fn=chat,
    title="My Agent",
    description="Chat with your AI agent."
)

if __name__ == "__main__":
    demo.launch()
'''

print("Failure handling app:")
print(FAILURE_HANDLING_APP)

### 💡 Why This Works

Gradio's generator pattern means any `yield` — including an error message — updates the chat window. Catching exceptions at the chat function level keeps errors contained: the user sees a human-readable message, the app stays running, and the next message works normally. Expose the error type but not the full stack trace.

---

## 🚀 Part 6: Deploying to Hugging Face Spaces

Hugging Face is a central platform for the AI community; a Space is a free hosted environment for Gradio apps. To deploy:

1. Sign in at [huggingface.co](https://huggingface.co)
2. Click **New Space**, give it a name, and choose **Gradio** as the SDK
3. Upload your files via the web UI drag-and-drop, or `git push` to the Space's repo
4. Add your `OPENAI_API_KEY` under **Settings → Repository secrets**
5. The Space builds automatically and gets a public URL

Your agent gets a shareable link as soon as the build completes.

**The minimum file set:**

```
my_agent_space/
├── app.py              # your Gradio app (the chat function + demo.launch())
├── requirements.txt    # openai-agents, gradio
└── .gitignore          # include .env
```

**No `.env` file** — on Hugging Face Spaces, secrets go in the Space settings under "Repository secrets", not in a file you commit. Your code reads them the same way: `os.getenv("OPENAI_API_KEY")`.

In [ ]:
# Deployment file templates
# --------------------------

APP_PY = '''
# app.py — complete Gradio app for Hugging Face Spaces
import os
import gradio as gr
from agents import Agent, Runner
from agents.stream_events import RawResponsesStreamEvent
from openai.types.responses import ResponseTextDeltaEvent

MODEL = os.getenv("MODEL", "gpt-5-mini")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant.",
    model=MODEL
)


async def chat(message: str, history: list):
    try:
        response_text = ""

        result = Runner.run_streamed(agent, input=message)

        async for event in result.stream_events():
            if isinstance(event, RawResponsesStreamEvent):
                data = event.data
                if isinstance(data, ResponseTextDeltaEvent):
                    response_text += data.delta
                    yield response_text
    except Exception as e:
        yield f"⚠️ Something went wrong. Please try again. (Error: {type(e).__name__})"


demo = gr.ChatInterface(
    fn=chat,
    title="My Agent",
    description="Chat with your AI agent."
)

demo.launch()
'''

REQUIREMENTS_TXT = '''
openai-agents
gradio
'''

print("app.py:")
print(APP_PY)
print("requirements.txt:")
print(REQUIREMENTS_TXT)

### ✅ Deployment Checklist

Note: `APP_PY` uses `os.getenv()` for `MODEL` instead of importing from `config.py`. We're packaging this as a single `app.py` for deployment simplicity — you can still use the Lesson 30 module structure on Spaces by uploading those files alongside `app.py`. The change here is reading `MODEL` from `os.getenv()` so it can be set via Space secrets rather than a local `.env` file.

This template combines streaming and failure handling. If your agent uses tools, copy the `RunItemStreamEvent` block from Part 4 into this `chat()` function so tool activity is visible in the UI.

Before pushing to Hugging Face Spaces:

- `app.py` calls `demo.launch()` with no arguments — Spaces handles the port and host

- `requirements.txt` lists all dependencies — Spaces installs them on startup

- `OPENAI_API_KEY` is set as a Repository Secret in Space settings, not in a committed file

- `.env` is in `.gitignore` — never commit API keys

- Test locally with `python app.py` before pushing

---

## 💪 Practice Exercises

### Exercise 1: Streaming Chat App

*Covers: Gradio — building a streaming chat interface*

Build a streaming Gradio app around one of the agents you built in this course.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Streaming Chat App
# --------------------------------------------------------------
# Objective: Wrap an existing agent in a streaming Gradio interface.

# TODO 1: Choose one agent from earlier in the course
#          (the customer service agent from Lesson 26, the MCP assistant from Lesson 29,
#          or any agent you built in a practice exercise)

# TODO 2: Create a streaming chat function using Runner.run_streamed()
#          that yields response_text on each text delta

# TODO 3: Wrap it in gr.ChatInterface with a title and description

# TODO 4: Add a try/except block so failures show a friendly message

# TODO 5: Test it by running demo.launch() and chatting with the agent

# --- Write your code below this line ---

### Exercise 2: Deploy to Hugging Face Spaces

*Covers: end-to-end deployment to Hugging Face Spaces*

Take the app from Exercise 1 and prepare it for deployment.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Prepare for Deployment
# --------------------------------------------------------------
# Objective: Create the files needed to deploy your agent to HF Spaces.

# TODO 1: Create app.py with your streaming chat function
#            — no .env loading, reads OPENAI_API_KEY from os.getenv()
#            — calls demo.launch() with no arguments

# TODO 2: Create requirements.txt with:
#            openai-agents
#            gradio

# TODO 3: Create .gitignore with .env included

# TODO 4: (Optional) Create a Hugging Face Space, add OPENAI_API_KEY
#            as a Repository Secret, and push your files
#            Your agent will be live at: https://huggingface.co/spaces/<your-username>/<space-name>

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Gradio turns an agent into a chat app in minutes:**

- `gr.ChatInterface` handles the UI and visible chat history

- The agent is still stateless unless you pass history into the input or use sessions (Lesson 19)

- Your chat function just takes `message` and `history` — return a string or yield partial strings
<br>
<br>

**Streaming requires one change and one bridge:**

- Change `Runner.run()` to `Runner.run_streamed()`

- Bridge: accumulate text deltas, `yield` the growing string on each delta

- Gradio updates the UI on every `yield` automatically
<br>
<br>

**Tool visibility and failure handling keep the app feeling solid:**

- Yield a status message when a tool call starts — users see something instead of silence

- Tool failures and model errors are caught at the app level — yield a friendly message, never a stack trace
<br>
<br>

**Deployment to Hugging Face Spaces needs three files:**

- `app.py` — your Gradio app, calls `demo.launch()` with no arguments

- `requirements.txt` — dependencies, Spaces installs them on startup

- `OPENAI_API_KEY` as a Repository Secret — never commit API keys

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-32-deploying-with-gradio)

---

## 🎉 Course Complete

You've gone from a 5-line agent to a deployed chat interface — with tools, memory, guardrails, multi-agent orchestration, human oversight, MCP integration, and a live public URL.

**What you've built:**

- Four complete capstone projects: Research Agent, Multi-Agent Research Team, Production Customer Service Agent, MCP-Powered Personal Assistant

- A production-readiness toolkit: tools, structured outputs, sessions, guardrails, tracing, MCP integration, and deployment

- The judgment to choose the right level of complexity — and the habit of justifying it with data

The OpenAI Agents SDK is moving fast. Keep an eye on the official documentation at `openai.github.io/openai-agents-python` — new capabilities appear regularly.

**Build something. Evaluate it. Improve it. That's the loop.**

---